# RunnableWithMessageHistory에 ChatMessageHistory추가
### 이전 대화를 기억하는 Chain 생성방법

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 이전 대화내용을 기억하는 multi-turn Chain

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering ChatBot입니다. 주어진 질문에 대한 답변을 제공해주세요",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "#Question:\n{question}"),
    ]
)

llm = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | llm | StrOutputParser()

대화를 기록하는 체인 생성(`chain_with_history`)

In [22]:
# 세션 기록을 저장할 딕셔너리
store = {}

# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 Session ID]: {session_ids}")
    if session_ids not in store :   # Session ID가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]   # 해당 session ID에 대한 세션 기록 반환

In [23]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,   # 세션 기록을 가져오는 함수
    input_messages_key="question",   # 사용자의 질문이 템플릿 변수에 들어갈 key
    history_messages_key="chat_history"   # 기록 메세지 key
)

In [24]:
chain_with_history.invoke(
    # 질문 입력
    {"question":"내 이름은 희영이야."},
    # Session ID 기준으로 대화 기록
    config = {"configurable": {"session_id":"id_01"}},
)

[대화 Session ID]: id_01


'안녕하세요, 희영님! 만나서 반갑습니다. 어떻게 도와드릴까요?'

In [25]:
chain_with_history.invoke(
    # 질문 입력
    {"question":"내 이름이 뭐라고?"},
    # Session ID 기준으로 대화 기록
    config = {"configurable": {"session_id":"id_01"}},
)

[대화 Session ID]: id_01


'당신의 이름은 희영입니다.'

In [26]:
chain_with_history.invoke(
    # 질문 입력
    {"question":"내 이름이 뭐라고?"},
    # Session ID 기준으로 대화 기록
    config = {"configurable": {"session_id":"id_02"}},
)

[대화 Session ID]: id_02


'죄송하지만, 당신의 이름은 알 수 없습니다. 사용자 정보에 접근할 수 없기 때문입니다. 어떻게 도와드릴까요?'